In [1]:
from pyspark.sql import SparkSession
import getpass 
username=getpass.getuser()
spark=SparkSession. \
    builder. \
    config('spark.ui.port','0'). \
    config("spark.sql.warehouse.dir", f"/user/itv023868/warehouse"). \
    config('spark.shuffle.useOldFetchProtocol', 'true'). \
    enableHiveSupport(). \
    master('yarn'). \
    getOrCreate()

#### 1. create a dataframe with proper datatypes 

In [2]:
customer_schema = 'member_id string, emp_title string, emp_length string, home_ownership string, annual_inc float, addr_state string, zip_code string, country string, grade string, sub_grade string, verification_status string, tot_hi_cred_lim float, application_type string, annual_inc_joint float, verification_status_joint string'

In [3]:
customers_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(customer_schema) \
.load("/public/trendytech/lendingclubproject/raw/customers_data_csv")

In [4]:
customers_raw_df

member_id,emp_title,emp_length,home_ownership,annual_inc,addr_state,zip_code,country,grade,sub_grade,verification_status,tot_hi_cred_lim,application_type,annual_inc_joint,verification_status_joint
b59d80da191f5b573...,null,null,RENT,50000.0,OR,973xx,USA,A,A5,Source Verified,8600.0,Individual,null,null
202d9f56ecb7c3bc9...,police officer,7 years,OWN,85000.0,TX,799xx,USA,A,A5,Source Verified,272384.0,Individual,null,null
e5a140c0922b554b9...,community living ...,6 years,RENT,48000.0,NY,146xx,USA,B,B2,Source Verified,85092.0,Individual,null,null
e12aefc548f750777...,Office,10+ years,OWN,33000.0,CT,067xx,USA,F,F1,Verified,7100.0,Individual,null,null
1b3a50d854fbbf97e...,Special Tooling I...,10+ years,MORTGAGE,81000.0,TX,791xx,USA,E,E5,Verified,190274.0,Individual,null,null
1c4329e5f17697127...,Mine ops tech 6,2 years,MORTGAGE,68000.0,AZ,855xx,USA,C,C3,Not Verified,182453.0,Individual,null,null
5026c86ad983175eb...,caregiver,4 years,RENT,76020.0,WA,993xx,USA,C,C2,Source Verified,15308.0,Individual,null,null
9847d8c1e9d0b2084...,null,null,OWN,65000.0,IL,624xx,USA,E,E3,Verified,128800.0,Individual,null,null
8340dbe1adea41fb4...,Vice President Re...,8 years,MORTGAGE,111000.0,CT,063xx,USA,A,A1,Not Verified,343507.0,Individual,null,null
d4de0de3ab7d79ad4...,FOREMAN,10+ years,MORTGAGE,67000.0,WA,992xx,USA,G,G2,Verified,211501.0,Individual,null,null


In [5]:
customers_raw_df.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: float (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- tot_hi_cred_lim: float (nullable = true)
 |-- application_type: string (nullable = true)
 |-- annual_inc_joint: float (nullable = true)
 |-- verification_status_joint: string (nullable = true)



#### 2. Rename a few columns

In [6]:
customer_df_renamed = customers_raw_df.withColumnRenamed("annual_inc", "annual_income") \
.withColumnRenamed("addr_state", "address_state") \
.withColumnRenamed("zip_code", "address_zipcode") \
.withColumnRenamed("country", "address_country") \
.withColumnRenamed("tot_hi_credit_lim", "total_high_credit_limit") \
.withColumnRenamed("annual_inc_joint", "join_annual_income")

In [7]:
customer_df_renamed

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,tot_hi_cred_lim,application_type,join_annual_income,verification_status_joint
b59d80da191f5b573...,null,null,RENT,50000.0,OR,973xx,USA,A,A5,Source Verified,8600.0,Individual,null,null
202d9f56ecb7c3bc9...,police officer,7 years,OWN,85000.0,TX,799xx,USA,A,A5,Source Verified,272384.0,Individual,null,null
e5a140c0922b554b9...,community living ...,6 years,RENT,48000.0,NY,146xx,USA,B,B2,Source Verified,85092.0,Individual,null,null
e12aefc548f750777...,Office,10+ years,OWN,33000.0,CT,067xx,USA,F,F1,Verified,7100.0,Individual,null,null
1b3a50d854fbbf97e...,Special Tooling I...,10+ years,MORTGAGE,81000.0,TX,791xx,USA,E,E5,Verified,190274.0,Individual,null,null
1c4329e5f17697127...,Mine ops tech 6,2 years,MORTGAGE,68000.0,AZ,855xx,USA,C,C3,Not Verified,182453.0,Individual,null,null
5026c86ad983175eb...,caregiver,4 years,RENT,76020.0,WA,993xx,USA,C,C2,Source Verified,15308.0,Individual,null,null
9847d8c1e9d0b2084...,null,null,OWN,65000.0,IL,624xx,USA,E,E3,Verified,128800.0,Individual,null,null
8340dbe1adea41fb4...,Vice President Re...,8 years,MORTGAGE,111000.0,CT,063xx,USA,A,A1,Not Verified,343507.0,Individual,null,null
d4de0de3ab7d79ad4...,FOREMAN,10+ years,MORTGAGE,67000.0,WA,992xx,USA,G,G2,Verified,211501.0,Individual,null,null


In [8]:
from pyspark.sql.functions import current_timestamp

#### 3. insert a new column named as ingestion date(current time)

In [9]:
customers_df_ingestd = customer_df_renamed.withColumn("ingest_date", current_timestamp())

In [10]:
customers_df_ingestd

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,tot_hi_cred_lim,application_type,join_annual_income,verification_status_joint,ingest_date
b59d80da191f5b573...,null,null,RENT,50000.0,OR,973xx,USA,A,A5,Source Verified,8600.0,Individual,null,null,2026-01-19 14:14:...
202d9f56ecb7c3bc9...,police officer,7 years,OWN,85000.0,TX,799xx,USA,A,A5,Source Verified,272384.0,Individual,null,null,2026-01-19 14:14:...
e5a140c0922b554b9...,community living ...,6 years,RENT,48000.0,NY,146xx,USA,B,B2,Source Verified,85092.0,Individual,null,null,2026-01-19 14:14:...
e12aefc548f750777...,Office,10+ years,OWN,33000.0,CT,067xx,USA,F,F1,Verified,7100.0,Individual,null,null,2026-01-19 14:14:...
1b3a50d854fbbf97e...,Special Tooling I...,10+ years,MORTGAGE,81000.0,TX,791xx,USA,E,E5,Verified,190274.0,Individual,null,null,2026-01-19 14:14:...
1c4329e5f17697127...,Mine ops tech 6,2 years,MORTGAGE,68000.0,AZ,855xx,USA,C,C3,Not Verified,182453.0,Individual,null,null,2026-01-19 14:14:...
5026c86ad983175eb...,caregiver,4 years,RENT,76020.0,WA,993xx,USA,C,C2,Source Verified,15308.0,Individual,null,null,2026-01-19 14:14:...
9847d8c1e9d0b2084...,null,null,OWN,65000.0,IL,624xx,USA,E,E3,Verified,128800.0,Individual,null,null,2026-01-19 14:14:...
8340dbe1adea41fb4...,Vice President Re...,8 years,MORTGAGE,111000.0,CT,063xx,USA,A,A1,Not Verified,343507.0,Individual,null,null,2026-01-19 14:14:...
d4de0de3ab7d79ad4...,FOREMAN,10+ years,MORTGAGE,67000.0,WA,992xx,USA,G,G2,Verified,211501.0,Individual,null,null,2026-01-19 14:14:...


#### 4. Remove complete duplicate rows

In [11]:
customers_df_ingestd.count()

2260701

In [12]:
customers_distinct = customers_df_ingestd.distinct()

In [13]:
customers_distinct.count()

2260638

In [14]:
customers_distinct.createOrReplaceTempView("customers")

In [15]:
spark.sql("select * from customers")

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,tot_hi_cred_lim,application_type,join_annual_income,verification_status_joint,ingest_date
7a14325bf05240550...,Carryout/Receiving,< 1 year,RENT,12000.0,OH,458xx,USA,C,C4,Source Verified,5000.0,Individual,null,null,2026-01-19 14:15:...
1439a21eda4e41dee...,Warehouse manager,3 years,MORTGAGE,102000.0,CA,932xx,USA,A,A1,Source Verified,424749.0,Individual,null,null,2026-01-19 14:15:...
94f316f45de093459...,manager,10+ years,RENT,79000.0,WA,983xx,USA,B,B3,Source Verified,86333.0,Individual,null,null,2026-01-19 14:15:...
e152523bbcfa03b1c...,Managing Director,1 year,RENT,175000.0,FL,320xx,USA,A,A1,Source Verified,96853.0,Individual,null,null,2026-01-19 14:15:...
fc4eecbba0d97bff5...,Flight Attendant,2 years,RENT,36000.0,MI,494xx,USA,A,A1,Not Verified,58187.0,Individual,null,null,2026-01-19 14:15:...
91594e62966de8716...,Food service,< 1 year,MORTGAGE,72000.0,CT,067xx,USA,C,C1,Verified,388912.0,Joint App,132000.0,Verified,2026-01-19 14:15:...
367029687a66efc2a...,Operations manager,4 years,RENT,100000.0,NJ,070xx,USA,B,B4,Source Verified,68605.0,Individual,null,null,2026-01-19 14:15:...
ef7811dc557e4c5c4...,MD,3 years,MORTGAGE,250000.0,MD,217xx,USA,B,B2,Source Verified,216436.0,Individual,null,null,2026-01-19 14:15:...
cb888e77a803d2193...,Patient Accounts Rep,3 years,OWN,30000.0,PA,178xx,USA,A,A2,Verified,160822.0,Individual,null,null,2026-01-19 14:15:...
91e459947b0193043...,machinist,10+ years,MORTGAGE,80000.0,MI,483xx,USA,B,B3,Not Verified,225647.0,Individual,null,null,2026-01-19 14:15:...


#### 5. Remove the rows where annual_income is null

In [16]:
spark.sql("select count(*) from customers where annual_income is null")

count(1)
5


In [17]:
customers_income_filtered = spark.sql("select * from customers where annual_income is not null")

In [18]:
customers_income_filtered.createOrReplaceTempView("customers")

In [19]:
spark.sql("select count(*) from customers where annual_income is null")

count(1)
0


### 6. convert emp_length to integer

In [20]:
spark.sql("select distinct(emp_length) from customers")

emp_length
9 years
5 years
null
1 year
2 years
7 years
8 years
4 years
6 years
3 years


In [21]:
from pyspark.sql.functions import regexp_replace, col

In [22]:
customers_emplength_cleaned = customers_income_filtered.withColumn("emp_length", regexp_replace(col("emp_length"), "(\D)",""))

In [23]:
customers_emplength_cleaned

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,tot_hi_cred_lim,application_type,join_annual_income,verification_status_joint,ingest_date
c9ff766b4e18df87c...,Legal Assistant,9,MORTGAGE,85000.0,DC,200xx,USA,D,D1,Not Verified,472180.0,Individual,null,null,2026-01-19 14:16:...
a6580a620311d89c4...,Analyst,1,RENT,108000.0,NY,100xx,USA,C,C5,Source Verified,14800.0,Individual,null,null,2026-01-19 14:16:...
1f44e8995b5e6f7d3...,tankerman level 4,8,MORTGAGE,73200.0,TX,781xx,USA,C,C5,Source Verified,232003.0,Individual,null,null,2026-01-19 14:16:...
7e14dce1897949c4d...,null,null,MORTGAGE,43000.0,MI,483xx,USA,B,B2,Not Verified,258558.0,Individual,null,null,2026-01-19 14:16:...
630bef604e1f1534a...,Account Manager,10,OWN,98000.0,NJ,076xx,USA,B,B3,Not Verified,444000.0,Individual,null,null,2026-01-19 14:16:...
162e50c24692bf6cf...,Legal assistant,6,MORTGAGE,65000.0,TX,751xx,USA,B,B4,Not Verified,113992.0,Individual,null,null,2026-01-19 14:16:...
80a5730f76079a359...,PMO & Governance ...,3,RENT,125000.0,NY,101xx,USA,D,D2,Verified,56300.0,Individual,null,null,2026-01-19 14:16:...
9ec4ec71d767555f8...,Sommelier / Lead ...,4,RENT,75000.0,MS,395xx,USA,C,C4,Not Verified,51522.0,Individual,null,null,2026-01-19 14:16:...
f165b63cde8ae5afb...,Staff II IT Auditor,1,OWN,101000.0,TX,782xx,USA,F,F2,Source Verified,295907.0,Individual,null,null,2026-01-19 14:16:...
85d6c710fbcc4d25b...,Wire Technician,1,OWN,50000.0,NC,275xx,USA,C,C2,Not Verified,304189.0,Individual,null,null,2026-01-19 14:16:...


In [24]:
customers_emplength_cleaned.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_income: float (nullable = true)
 |-- address_state: string (nullable = true)
 |-- address_zipcode: string (nullable = true)
 |-- address_country: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- tot_hi_cred_lim: float (nullable = true)
 |-- application_type: string (nullable = true)
 |-- join_annual_income: float (nullable = true)
 |-- verification_status_joint: string (nullable = true)
 |-- ingest_date: timestamp (nullable = false)



In [25]:
customers_emplength_casted = customers_emplength_cleaned.withColumn("emp_length", customers_emplength_cleaned.emp_length.cast('int'))

In [26]:
customers_emplength_casted

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,tot_hi_cred_lim,application_type,join_annual_income,verification_status_joint,ingest_date
fb01fb66a9145dfdf...,Project Manager,10,MORTGAGE,100000.0,WA,980xx,USA,F,F2,Source Verified,561059.0,Joint App,136643.0,Source Verified,2026-01-19 14:16:...
560f2f5ffee8fc87e...,Assistant Manager,1,RENT,60000.0,NY,104xx,USA,C,C5,Not Verified,13100.0,Individual,null,null,2026-01-19 14:16:...
7607c103533069135...,Aircraft Maintena...,1,OWN,215000.0,NY,109xx,USA,A,A3,Source Verified,410300.0,Individual,null,null,2026-01-19 14:16:...
e2838df06260c7fc5...,Room Attendant,4,RENT,65000.0,NY,104xx,USA,D,D5,Source Verified,56100.0,Individual,null,null,2026-01-19 14:16:...
e20f46e7a3c00b64e...,null,null,RENT,42000.0,TX,750xx,USA,D,D2,Verified,26271.0,Individual,null,null,2026-01-19 14:16:...
9e417dd5e376575c9...,Technical Recruiter,2,RENT,50500.0,FL,328xx,USA,D,D2,Source Verified,125486.0,Individual,null,null,2026-01-19 14:16:...
64e39a8730e9ae91b...,Payroll,10,MORTGAGE,65000.0,TX,775xx,USA,B,B1,Not Verified,78500.0,Individual,null,null,2026-01-19 14:16:...
e876736122fd51953...,"Director, IT Clie...",1,MORTGAGE,185000.0,AZ,852xx,USA,A,A5,Not Verified,686566.0,Individual,null,null,2026-01-19 14:16:...
0b8494c31c7c92807...,Quality Control C...,10,MORTGAGE,46000.0,NE,680xx,USA,C,C2,Verified,191087.0,Individual,null,null,2026-01-19 14:16:...
1849e73e74982f9a6...,Team leader,10,OWN,75000.0,TN,377xx,USA,A,A4,Not Verified,62249.0,Individual,null,null,2026-01-19 14:16:...


In [27]:
customers_emplength_casted.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: integer (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_income: float (nullable = true)
 |-- address_state: string (nullable = true)
 |-- address_zipcode: string (nullable = true)
 |-- address_country: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- tot_hi_cred_lim: float (nullable = true)
 |-- application_type: string (nullable = true)
 |-- join_annual_income: float (nullable = true)
 |-- verification_status_joint: string (nullable = true)
 |-- ingest_date: timestamp (nullable = false)



#### 7. we need to replace all the nulls in emp_length column with average of this column

In [28]:
customers_emplength_casted.filter("emp_length is null").count()

146903

In [29]:
customers_emplength_casted.createOrReplaceTempView("customers")

In [30]:
avg_emp_length = spark.sql("select floor(avg(emp_length)) as avg_emp_length from customers").collect()

In [31]:
print(avg_emp_length)

[Row(avg_emp_length=6)]


In [32]:
avg_emp_duration = avg_emp_length[0][0]

In [33]:
print(avg_emp_duration)

6


In [34]:
customers_emplength_replaced = customers_emplength_casted.na.fill(avg_emp_duration, subset=['emp_length'])

In [35]:
customers_emplength_replaced

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,tot_hi_cred_lim,application_type,join_annual_income,verification_status_joint,ingest_date
635e73febcdee5b56...,Vice President,10,MORTGAGE,450000.0,MA,024xx,USA,B,B4,Verified,320583.0,Individual,null,null,2026-01-19 14:16:...
d28041b29f2b63478...,Maintenance Super...,7,RENT,38258.99,NV,891xx,USA,D,D5,Verified,55305.0,Individual,null,null,2026-01-19 14:16:...
51229c3d76b4dc71d...,manager,10,OWN,100000.0,WV,254xx,USA,F,F1,Not Verified,258592.0,Individual,null,null,2026-01-19 14:16:...
8cb78a6711d934f83...,SOCIAL INSURANCE ...,5,RENT,95900.0,CA,945xx,USA,A,A4,Not Verified,112705.0,Individual,null,null,2026-01-19 14:16:...
abf4010ed4699436b...,Pharmacist,6,MORTGAGE,138000.0,NC,280xx,USA,A,A5,Source Verified,320775.0,Individual,null,null,2026-01-19 14:16:...
de7de9731c60ebe64...,Paralegal,1,RENT,25000.0,TX,785xx,USA,B,B3,Verified,118778.0,Individual,null,null,2026-01-19 14:16:...
eaba3714e423ef545...,Owner,8,RENT,113633.0,NJ,070xx,USA,A,A2,Source Verified,55730.0,Individual,null,null,2026-01-19 14:16:...
b6c1ebc666476adc8...,Custodian,1,MORTGAGE,64000.0,IN,473xx,USA,B,B5,Not Verified,88298.0,Individual,null,null,2026-01-19 14:16:...
1c7a8ca2c188c7adb...,Lead Analyst,10,MORTGAGE,72567.0,TX,750xx,USA,A,A5,Verified,219432.0,Individual,null,null,2026-01-19 14:16:...
3a070f8a5fe4397cf...,Sales,3,RENT,40000.0,NJ,070xx,USA,E,E1,Source Verified,26200.0,Individual,null,null,2026-01-19 14:16:...


In [36]:
customers_emplength_replaced.filter("emp_length is null").count()

0

#### 8. Clean the address_state(it should be 2 characters only),replace all others with NA

In [37]:
customers_emplength_replaced.createOrReplaceTempView("customers")

In [38]:
spark.sql("select distinct(address_state) from customers")

address_state
Helping Kenya's D...
175 (total projec...
223xx
SC
AZ
"so Plan """"C"""" is ..."
I am 56 yrs. old ...
financially I mad...
but no one will l...
LA


In [39]:
spark.sql("select count(address_state) from customers where length(address_state)>2")

count(address_state)
254


In [40]:
from pyspark.sql.functions import when, col, length

In [41]:
customers_state_cleaned = customers_emplength_replaced.withColumn(
    "address_state",
    when(length(col("address_state"))> 2, "NA").otherwise(col("address_state"))
)

In [42]:
customers_state_cleaned

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,tot_hi_cred_lim,application_type,join_annual_income,verification_status_joint,ingest_date
680d41b22550feffe...,Programmer/Analyst,10,RENT,39000.0,CA,941xx,USA,A,A5,Source Verified,23100.0,Individual,null,null,2026-01-19 14:17:...
0bd1b4a2394c518f3...,supervisor,10,RENT,47000.0,MA,184xx,USA,E,E3,Verified,41787.0,Individual,null,null,2026-01-19 14:17:...
cd8ee7b5c9bc5e0fe...,Financial Advisor,2,MORTGAGE,200000.0,TX,775xx,USA,B,B2,Verified,473593.0,Individual,null,null,2026-01-19 14:17:...
18b54eab581144672...,Maintenance lead,8,MORTGAGE,40000.0,MI,486xx,USA,F,F5,Verified,149212.0,Individual,null,null,2026-01-19 14:17:...
5e18bc1f6daf2e451...,Manager,10,RENT,46112.0,VA,201xx,USA,E,E4,Verified,22200.0,Individual,null,null,2026-01-19 14:17:...
4846aa9452f994930...,HVAC Tech,7,RENT,95000.0,MI,493xx,USA,G,G5,Verified,103545.0,Individual,null,null,2026-01-19 14:17:...
9ad7f61a97e7e4975...,Painter,2,RENT,27000.0,FL,330xx,USA,D,D3,Source Verified,14307.0,Individual,null,null,2026-01-19 14:17:...
4b57441c96b6a54f9...,Vehicle Inventory,4,MORTGAGE,57000.0,CA,953xx,USA,A,A5,Source Verified,241200.0,Individual,null,null,2026-01-19 14:17:...
692be0aed4b5765d8...,Technical,10,OWN,180000.0,OH,430xx,USA,A,A4,Not Verified,158833.0,Individual,null,null,2026-01-19 14:17:...
7922cbd9f157e3575...,Testing Coordinator,1,OWN,18000.0,NY,141xx,USA,A,A2,Source Verified,29000.0,Individual,null,null,2026-01-19 14:17:...


In [43]:
customers_state_cleaned.select("address_state").distinct()

address_state
SC
AZ
LA
MN
NJ
DC
OR
NA
VA
null


In [45]:
customers_state_cleaned.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv023868/lendingclubproject/raw/cleaned/customers_parquet") \
.save()

In [46]:
customers_state_cleaned.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "/user/itv023868/lendingclubproject/raw/cleaned/customers_csv") \
.save()